# FrozenLake Game - Google Colab Version

Este notebook implementa o jogo FrozenLake usando matplotlib para visualização e ipywidgets para controle interativo. O objetivo é controlar um agente para alcançar o objetivo (célula verde) sem cair em buracos (células azuis).

## Como jogar:
- Use os botões **↑ ↓ ← →** para mover o agente (círculo vermelho)
- Evite os buracos (azuis)
- Alcance o objetivo (verde)
- Ao cair ou vencer, clique em **Reiniciar**
- Marque **Gelo escorregadio** para um desafio: o gelo nem sempre te leva para onde você mandou

> Aqui **você é o agente**: observa o estado (a sua casa), escolhe a ação (um botão) e o ambiente responde. Rode todas as células e vá direto para a seção 5.

## 1. Importar Bibliotecas Necessárias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import ipywidgets as widgets
from IPython.display import display, clear_output

## 2. Definir o Ambiente FrozenLake

In [ ]:
# Definindo o tamanho do ambiente (tabuleiro 4x4)
n_rows, n_cols = 4, 4

# Configurações de células:
# 0 = caminho livre (branco)
# 1 = buraco/gelo frágil (azul)
# 2 = objetivo/meta (verde)

frozen_lake_map = np.array([
    [0, 0, 0, 0],
    [0, 1, 0, 1],
    [0, 0, 0, 1],
    [0, 1, 0, 2]
])

# Posição inicial do agente (canto superior esquerdo)
agent_position = [0, 0]

# Estatísticas do jogo
game_stats = {
    'moves': 0,
    'status': 'Jogando...',
    'won': False,
    'lost': False
}

print("Mapa FrozenLake criado:")
print("0 = Caminho livre | 1 = Buraco | 2 = Objetivo")
print("\nMapa:")
print(frozen_lake_map)

## 3. Criar Funções de Visualização

In [ ]:
# @title Função de desenho do tabuleiro (rode e siga em frente)
def draw_frozen_lake(agent_pos, ax=None):
    """
    Desenha o tabuleiro do FrozenLake com matplotlib
    
    Parâmetros:
    - agent_pos: posição [linha, coluna] do agente
    - ax: eixo matplotlib para desenhar
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    else:
        ax.clear()
    
    # Cores para cada tipo de célula
    colors = {
        0: 'white',      # caminho livre
        1: '#4A90E2',    # buraco (azul)
        2: '#7ED321'     # objetivo (verde)
    }
    
    # Desenhar cada célula do tabuleiro
    for row in range(n_rows):
        for col in range(n_cols):
            cell_value = frozen_lake_map[row, col]
            color = colors[cell_value]
            
            # Desenhar retângulo da célula
            rect = plt.Rectangle((col, n_rows - row - 1), 1, 1, 
                                 linewidth=2, edgecolor='black', 
                                 facecolor=color)
            ax.add_patch(rect)
            
            # Adicionar label para a célula se necessário
            if cell_value == 2:  # Objetivo
                ax.text(col + 0.5, n_rows - row - 1 + 0.5, '★',
                       ha='center', va='center', fontsize=20)
            elif cell_value == 1:  # Buraco
                ax.text(col + 0.5, n_rows - row - 1 + 0.5, '✗',
                       ha='center', va='center', fontsize=16, color='white')
    
    # Desenhar o agente (círculo vermelho)
    agent_x = agent_pos[1] + 0.5
    agent_y = n_rows - agent_pos[0] - 1 + 0.5
    circle = Circle((agent_x, agent_y), 0.2, color='red', zorder=10)
    ax.add_patch(circle)
    
    # Adicionar indicador de posição do agente na célula
    ax.text(agent_x, agent_y - 0.35, f"({agent_pos[0]},{agent_pos[1]})",
           ha='center', va='center', fontsize=9, 
           bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    # Configurar eixos
    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Título com informações detalhadas
    status_color = 'green' if game_stats['won'] else 'red' if game_stats['lost'] else 'black'
    title = f"FrozenLake - Posição Atual: ({agent_pos[0]}, {agent_pos[1]}) | Movimentos: {game_stats['moves']}"
    ax.set_title(title, fontsize=13, fontweight='bold', color=status_color, pad=15)
    
    # Adicionar status na parte inferior do gráfico
    status_text = game_stats['status']
    ax.text(n_cols / 2, -0.5, status_text, 
           ha='center', va='top', fontsize=11, fontweight='bold',
           color=status_color)
    
    return ax

# Criar a figura inicialmente
fig, ax = plt.subplots(figsize=(6, 6))
draw_frozen_lake(agent_position, ax)
plt.tight_layout()
plt.show()

print("✓ Funções de visualização criadas!")

## 4. Implementar Lógica de Movimento

In [ ]:
def move_agent(action):
    """
    Move o agente baseado na ação fornecida
    
    Parâmetros:
    - action: string ('up', 'down', 'left', 'right')
    
    Retorna:
    - True se o movimento foi realizado, False caso contrário
    """
    global agent_position, game_stats
    
    old_position = agent_position.copy()
    
    if action == 'up' and agent_position[0] > 0:
        agent_position[0] -= 1
    elif action == 'down' and agent_position[0] < n_rows - 1:
        agent_position[0] += 1
    elif action == 'left' and agent_position[1] > 0:
        agent_position[1] -= 1
    elif action == 'right' and agent_position[1] < n_cols - 1:
        agent_position[1] += 1
    else:
        return False
    
    # Aumentar contador de movimentos apenas se o movimento foi válido
    if agent_position != old_position:
        game_stats['moves'] += 1
    
    return agent_position != old_position

def check_game_state():
    """
    Verifica o estado do jogo e atualiza as informações
    """
    global game_stats
    
    cell_value = frozen_lake_map[agent_position[0], agent_position[1]]
    
    if cell_value == 1:  # Buraco
        game_stats['status'] = '✗ CAIU EM UM BURACO!'
        game_stats['lost'] = True
        return 'lost'
    elif cell_value == 2:  # Objetivo
        game_stats['status'] = '★ OBJETIVO ALCANÇADO!'
        game_stats['won'] = True
        return 'won'
    else:
        game_stats['status'] = 'Jogando...'
        game_stats['won'] = False
        game_stats['lost'] = False
        return 'playing'

def reset_game():
    """
    Reinicia o jogo
    """
    global agent_position, game_stats
    agent_position = [0, 0]
    game_stats = {
        'moves': 0,
        'status': 'Jogando...',
        'won': False,
        'lost': False
    }

print("Funções de movimento implementadas!")

## 5. Executar o Jogo Interativo

Use os botões abaixo para controlar o agente. Clique em **Reiniciar** para começar um novo jogo.

In [ ]:
# @title Interface do jogo — rode para jogar
# Criar a interface do jogo
output = widgets.Output()
game_display = widgets.Output()

# Botões de controle
btn_up = widgets.Button(description='↑', button_style='info', layout=widgets.Layout(width='50px'))
btn_down = widgets.Button(description='↓', button_style='info', layout=widgets.Layout(width='50px'))
btn_left = widgets.Button(description='←', button_style='info', layout=widgets.Layout(width='50px'))
btn_right = widgets.Button(description='→', button_style='info', layout=widgets.Layout(width='50px'))
btn_reset = widgets.Button(description='Reiniciar', button_style='warning', layout=widgets.Layout(width='100px'))
chk_escorregadio = widgets.Checkbox(value=False, description='Gelo escorregadio', indent=False)

LADOS = {'up': ['left', 'right'], 'down': ['left', 'right'], 'left': ['up', 'down'], 'right': ['up', 'down']}
SETAS = {'up': '↑', 'down': '↓', 'left': '←', 'right': '→'}

def acao_executada(acao_pedida):
    '''Gelo escorregadio (como no FrozenLake original): 1/3 vai para onde você pediu,
    1/3 escorrega para cada um dos lados.'''
    if not chk_escorregadio.value:
        return acao_pedida
    return np.random.choice([acao_pedida] + LADOS[acao_pedida])

# Layout dos botões
button_layout = widgets.VBox([
    widgets.HBox([widgets.Label(value='Movimento:', layout=widgets.Layout(width='70px')), 
                  widgets.Label(value='')]),
    widgets.HBox([widgets.Label(value=''), btn_up, widgets.Label(value='')],
                layout=widgets.Layout(justify_content='center')),
    widgets.HBox([btn_left, widgets.Label(value='', layout=widgets.Layout(width='50px')), btn_right],
                layout=widgets.Layout(justify_content='center')),
    widgets.HBox([widgets.Label(value=''), btn_down, widgets.Label(value='')],
                layout=widgets.Layout(justify_content='center')),
    widgets.HBox([btn_reset], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([chk_escorregadio], layout=widgets.Layout(justify_content='center'))
])

def update_display():
    """Atualiza a visualização do jogo com matplotlib"""
    with game_display:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6, 6))
        draw_frozen_lake(agent_position, ax)
        plt.tight_layout()
        plt.show()

def on_button_click(action):
    """Callback para cliques nos botões - exibe movimento e atualiza visualização"""
    def handler(button):
        if not game_stats['won'] and not game_stats['lost']:
            pos_antes = agent_position.copy()
            executada = acao_executada(action)
            move_agent(executada)
            
            # Mostrar o movimento realizado
            with output:
                clear_output(wait=True)
                if executada != action:
                    print(f"Escorregou! Você pediu {SETAS[action]}, o gelo te levou para {SETAS[executada]}")
                if agent_position != pos_antes:
                    print(f"Movimento: {executada.upper()}")
                    print(f"Posição anterior: ({pos_antes[0]}, {pos_antes[1]})")
                    print(f"Posição atual: ({agent_position[0]}, {agent_position[1]})")
                    print(f"Movimentos totais: {game_stats['moves']}")
                else:
                    print(f"Você não pode se mover para {executada.upper()} (limite do tabuleiro)")
            
            state = check_game_state()
            update_display()
            
            if state == 'won':
                with output:
                    print(f"\nPARABÉNS! Você alcançou o objetivo em {game_stats['moves']} movimentos!")
            elif state == 'lost':
                with output:
                    print(f"\nGAME OVER! Você caiu em um buraco!")
                    print(f"Clique em 'Reiniciar' para tentar novamente.")
    return handler

def on_reset_click(button):
    """Callback para o botão de reiniciar"""
    reset_game()
    update_display()
    with output:
        clear_output()
        print("Jogo reiniciado!")
        print("Posição inicial: (0, 0)")
        print("Use os botões para mover o agente (círculo vermelho)")

# Conectar callbacks aos botões
btn_up.on_click(on_button_click('up'))
btn_down.on_click(on_button_click('down'))
btn_left.on_click(on_button_click('left'))
btn_right.on_click(on_button_click('right'))
btn_reset.on_click(on_reset_click)

# Exibir interface
print("JOGO FROZENLAKE - GOOGLE COLAB")
print("=" * 50)
print("Use os botões para mover o agente (círculo vermelho)")
print("Objetivo: Alcançar a estrela verde (★)")
print("Evite: Os buracos azuis (✗)")
print("Reinicie: Use o botão 'Reiniciar' para começar de novo")
print("=" * 50 + "\n")

update_display()

display(button_layout)
display(output)

## 6. Informações Adicionais

### Sobre o FrozenLake
- **Objetivo**: Mover o agente (círculo vermelho) do canto superior esquerdo até o objetivo (★ verde)
- **Obstáculos**: Evitar os buracos (✗ azuis)
- **Movimentos**: Apenas 4 direções (cima, baixo, esquerda, direita)
- **Limites**: O agente não pode sair do tabuleiro 4x4

## 7. Para pensar

1. **Você é o agente.** Qual é o **estado** que você observa a cada jogada? Quais são as **ações**? Que **recompensa** você recebe — e quando?
2. Qual é o **menor número de movimentos** até o objetivo? Você seguiu uma regra na cabeça para chegar lá: essa regra é a sua **política**.
3. Ligue o **gelo escorregadio** e jogue umas 5 partidas. Sua estratégia mudou? Você ainda passaria ao lado de um buraco?
4. Com o gelo escorregadio, a mesma ação nem sempre leva à mesma casa. Que objeto do MDP descreve isso? (dica: P(s' | s, a), aula de MDP)
5. Um computador não "vê" o tabuleiro como você. Como ele poderia descobrir uma boa política só jogando muitas vezes e anotando o que deu certo?